# Experiment Runner with Automatic Data Loading

This notebook demonstrates the ExperimentRunner which:
1. **Loads experiment configuration from YAML** - including data loading config
2. **Automatically loads and parses data** - using configured loader and parser
3. **Runs multiple RAG configs** - either sequentially or in parallel
4. **Generates detailed reports** - per-query and aggregate metrics
5. **Compares results** - across different configurations

The experiment configuration in YAML specifies:
- Data source (HuggingFace dataset)
- Data parser (e.g., title_passage)
- RAG config directory
- Number of queries to evaluate
- Parallel execution settings

## 1. Setup & Dependencies

In [1]:
get_ipython().system('pip install datasets faiss-cpu sentence-transformers torch groq python-dotenv nltk pandas -q')

zsh:1: /Users/adityanarayan/aiml-talentsprint/rag-capstone-project/rag-foundry/venv/bin/pip: bad interpreter: /Users/adityanarayan/aiml-talentsprint/rag-capstone-project/real-world-rag-system/venv/bin/python: no such file or directory

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python3.11 -m pip install --upgrade pip


## 2. Imports

In [2]:
import os
from dotenv import load_dotenv

from experiment.experiment_config import ExperimentConfig
from experiment.experiment_runner import ExperimentRunner

load_dotenv(override=True)
print("HuggingFace token loaded:", bool(os.getenv("HF_TOKEN")))
print("Groq API key loaded:", bool(os.getenv("GROQ_API_KEY")))

/Users/adityanarayan/aiml-talentsprint/rag-capstone-project/rag-foundry/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


HuggingFace token loaded: True
Groq API key loaded: True


## 3. Load Experiment Configuration

The experiment configuration file specifies:
- **data_loader**: How to load data (HuggingFace with dataset_name, subset, split)
- **data_parser**: How to parse documents (title_passage)
- **config_dir**: Directory containing RAG pipeline configs
- **num_queries**: Number of queries to evaluate
- **parallel**: Whether to run configs in parallel

In [3]:
# Load experiment configuration from YAML
EXPERIMENT_CONFIG_PATH = "experiment_configs/covidqa_experiment.yaml"
experiment_config = ExperimentConfig.load(EXPERIMENT_CONFIG_PATH)

print("Experiment Configuration:")
print(f"  Config Dir:  {experiment_config.config_dir}")
print(f"  Report Dir:  {experiment_config.report_dir}")
print(f"  Cache Dir:   {experiment_config.cache_dir}")
print(f"  Num Queries: {experiment_config.end_index}")
print(f"  Parallel:    {experiment_config.parallel}")
print(f"  Max Workers: {experiment_config.max_workers}")
print(f"\nData Loader:")
print(f"  Type: {experiment_config.data_loader['type']}")
print(f"  Config: {experiment_config.data_loader['config']}")
print(f"\nData Parser:")
print(f"  Type: {experiment_config.data_parser}")

Experiment Configuration:
  Config Dir:  config
  Report Dir:  reports
  Cache Dir:   cache
  Num Queries: 246
  Parallel:    False
  Max Workers: 4

Data Loader:
  Type: huggingface
  Config: {'dataset_name': 'galileo-ai/ragbench', 'subset': 'covidqa', 'split': 'test', 'limit': 246}

Data Parser:
  Type: title_passage


## 4. Initialize Experiment Runner

The ExperimentRunner will:
- Create report directory if it doesn't exist
- Load RAG configs from the specified directory
- Load and parse data automatically based on YAML config

In [4]:
# Initialize experiment runner
runner = ExperimentRunner(experiment_config)
print("ExperimentRunner initialized")

ExperimentRunner initialized


import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

for report in reports:
    print(f"\n{'='*80}")
    print(f"Configuration: {report.config_name}")
    print(f"{'='*80}")
    
    # Display per-query results
    print("\nPer-Query Results:")
    display(report.sections[0].per_query)
    
    # Display aggregate statistics
    print("\nAggregate Statistics:")
    display(report.sections[0].summary)

In [5]:
# Load data automatically based on YAML configuration
print("Loading data based on experiment configuration...")
documents, raw_data = runner.load_data()

print(f"\n✅ Data loaded successfully!")
print(f"  Documents: {len(documents)} parsed documents")
print(f"  Raw Data:  {len(raw_data)} samples")

# Inspect first sample
first_sample = raw_data[0]
print(f"\nFirst Sample:")
print(f"  Question: {first_sample['question'][:100]}...")
print(f"  Documents: {len(first_sample['documents'])}")

Loading data based on experiment configuration...
Loading HuggingFace dataset: galileo-ai/ragbench/covidqa (test)...
Loaded 246 samples

✅ Data loaded successfully!
  Documents: 984 parsed documents
  Raw Data:  246 samples

First Sample:
  Question: Which viruses may not cause prolonged inflammation due to strong induction of antiviral clearance?...
  Documents: 4


## 6. Load RAG Pipeline Configs

Load all RAG pipeline configurations from the config directory specified in the experiment config.

In [6]:
# Load RAG pipeline configs
configs = runner.load_configs()

print(f"Loaded {len(configs)} RAG pipeline configurations:")
for cfg in configs:
    print(f"  - {cfg.name}")
    print(f"    Chunking: {cfg.chunking.type.value}")
    print(f"    Retrieval Pipeline: ")
    for search in cfg.retrieval.search.searches:
        print(f"      - {search.type.value}")
    print(f"    Reranker: {cfg.retrieval.rerank.type.value}")
    print(f"    Generation: {cfg.generation.config.model}")

Loaded 3 RAG pipeline configurations:
  - covidqa_high_recall_hybrid_v2
    Chunking: semantic
    Retrieval Pipeline: 
      - dense
      - sparse
    Reranker: cross_encoder
    Generation: llama-3.1-8b-instant
  - covidqa_hyde_hybrid
    Chunking: semantic
    Retrieval Pipeline: 
      - dense
      - sparse
    Reranker: cross_encoder
    Generation: llama-3.1-8b-instant
  - covidqa_multiquery_hybrid
    Chunking: semantic
    Retrieval Pipeline: 
      - dense
      - sparse
    Reranker: cross_encoder
    Generation: llama-3.1-8b-instant


## 7. Run Experiments

Run all RAG configurations on the loaded data. Each config will:
1. Build a vector index from the documents
2. Run queries against the index
3. Generate responses
4. Evaluate using TRACe metrics

Results are returned as PipelineRunResult objects.

In [ ]:
# Run experiments

print(f"Running {len(configs)} configurations from {experiment_config.start_index} to {experiment_config.end_index} queries...")
print(f"Parallel mode: {experiment_config.parallel}")

runs = runner.run(documents, raw_data)

print(f"\n✅ Experiments completed!")
print(f"  Ran {len(runs)} configurations")

for run in runs:
    print(f"  - {run.config.name}: {len(run.records)} queries")

Running 3 configurations from 0 to 246 queries...
Parallel mode: False
Loading embedding model: BAAI/bge-large-en-v1.5


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 8074.98it/s]


Loading reranker model: BAAI/bge-reranker-v2-m3


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 7282.49it/s]


Progress: 71/246 (28.9%) | QPS: 0.53 | ETA: 5.5m | Elapsed: 2.2m

...

Progress: 102/246 (41.5%) | QPS: 0.25 | ETA: 9.5m | Elapsed: 6.7m

2026-06-28 21:26:14,474 ERROR rag.pipeline.rag_pipeline: Query failed: All Groq API keys are currently rate limited.
--- Logging error ---
Traceback (most recent call last):
  File "/Users/adityanarayan/aiml-talentsprint/rag-capstone-project/rag-foundry/rag/pipeline/rag_pipeline.py", line 240, in query
    response = self.generator.generate(
               ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/adityanarayan/aiml-talentsprint/rag-capstone-project/rag-foundry/rag/generation/default_generation.py", line 73, in generate
    response = self.provider.generate(
               ^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/adityanarayan/aiml-talentsprint/rag-capstone-project/rag-foundry/providers/groq/groq_provider.py", line 116, in generate
    key_state = self._get_next_available_key()
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/adityanarayan/aiml-talentsprint/rag-capstone-project/rag-foundry/providers/groq/groq_provider.py", line 101, in _get_next_available_key
    raise AllKeys

Progress: 102/246 (41.5%) | QPS: 0.25 | ETA: 9.5m | Elapsed: 6.8m

2026-06-28 21:26:16,708 ERROR rag.pipeline.rag_pipeline: Query failed: All Groq API keys are currently rate limited.


Progress: 102/246 (41.5%) | QPS: 0.13 | ETA: 18.6m | Elapsed: 13.2m

In [11]:
for run in runs:
    print(run)

{'config_name': 'medcpt_semantic_hybrid_rerank_70b', 'config': RAGConfig(providers={'groq': ProviderConfig(type=<ProviderType.GROQ: 'groq'>, api_key_env='GROQ_API_KEY', params={'cooldown_seconds': 60})}, chunking=ChunkingConfig(type=<ChunkingType.SEMANTIC: 'semantic'>, config=SemanticChunkingConfig(embedding=EmbeddingConfig(type=<EmbeddingType.SENTENCE_TRANSFORMER: 'sentence_transformer'>, config=SentenceTransformerEmbeddingConfig(model_name='sentence-transformers/all-MiniLM-L6-v2', model=None, dimension=768)), max_words=100, similarity_threshold=0.8, overlap_sentences=1, similarity_window=5)), embedding=EmbeddingConfig(type=<EmbeddingType.MEDCPT: 'medcpt'>, config=MedCPTEmbeddingConfig(query_model_name='ncbi/MedCPT-Query-Encoder', article_model_name='ncbi/MedCPT-Article-Encoder', dimension=768)), vector_store=VectorStoreConfig(type=<VectorStoreType.FAISS: 'faiss'>, config=FaissVectorStoreConfig(dimension=768)), retrieval=RetrievalConfig(search=SearchPipelineConfig(searches=[SearchStra

## 8. Generate Reports

Generate detailed reports for each configuration including:
- Per-query table with all TRACe scores
- Aggregate statistics (mean, std, MAE)
- Comparison with ground truth

In [12]:
# Generate reports
print("Generating reports...")
reports = runner.generate_reports(runs)

print(f"\n✅ Reports generated!")
print(f"  Saved to: {experiment_config.report_dir}")

Generating reports...

✅ Reports generated!
  Saved to: reports


## 9. Display Reports

Display the generated reports with per-query and aggregate metrics.

In [13]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

for report in reports:
    print(f"\n{'='*80}")
    print(f"Configuration: {report.config_name}")
    print(f"{'='*80}")
    
    # Display per-query results
    print("\nPer-Query Results:")
    display(report.display())
    


Configuration: medcpt_semantic_hybrid_rerank_70b

Per-Query Results:


# RAG Multi-Config Evaluation Report

_Strategy: detailed_query_

## Config: `medcpt_semantic_hybrid_rerank_70b`

**name**: medcpt_semantic_hybrid_rerank_70b  •  **mode**: test  •  **providers**: {'groq': {'type': 'groq', 'api_key_env': 'GROQ_API_KEY', 'params': {'cooldown_seconds': 60}}}  •  **chunking**: {'type': 'semantic', 'config': {'embedding': {'type': 'sentence_transformer', 'config': {'model_name': 'sentence-transformers/all-MiniLM-L6-v2', 'model': None, 'dimension': 768}}, 'max_words': 100, 'similarity_threshold': 0.8, 'overlap_sentences': 1, 'similarity_window': 5}}  •  **embedding**: {'type': 'medcpt', 'config': {'query_model_name': 'ncbi/MedCPT-Query-Encoder', 'article_model_name': 'ncbi/MedCPT-Article-Encoder', 'dimension': 768}}  •  **vector_store**: {'type': 'faiss', 'config': {'dimension': 768}}  •  **retrieval**: {'search': {'searches': [{'type': 'dense', 'config': {'top_k': 20}}, {'type': 'sparse', 'config': {'top_k': 20}}]}, 'query_transform': None, 'fusion': {'type': 'rrf', 'config': {'top_k': 5, 'k': 60}}, 'rerank': {'type': 'cross_encoder', 'config': {'model_name': 'BAAI/bge-reranker-v2-m3', 'top_k': 5}}}  •  **generation**: {'strategy': 'default', 'provider': 'groq', 'config': {'model': 'llama-3.1-8b-instant', 'max_tokens': 1024, 'temperature': 0.7, 'system_prompt': None}}  •  **evaluation**: {'type': 'trace', 'provider': 'groq', 'config': {'model': 'llama-3.3-70b-versatile', 'max_tokens': 2000, 'temperature': 0.0}}  •  **cache**: {'enabled': True, 'cache_dir': 'cache'}  •  **start_index**: None  •  **end_index**: None  •  **logging_config**: {'enabled': True, 'level': 'INFO', 'show_progress': True}

...

### Per-query results

,query,retrieved_documents,answer,relevance_score__pred,relevance_score__gt,relevance_score__deviation,utilization_score__pred,utilization_score__gt,utilization_score__deviation,completeness_score__pred,completeness_score__gt,completeness_score__deviation,adherence_score__pred,adherence_score__gt,adherence_score__deviation
0,How many antigens could be detected by Liew's multiplex ELISA test?,[{'text': 'Antibody arrays for simultaneous multiple antigen quantification ...,"According to Document 1 and Document 2, Liew validated one multiplex ELISA f...",0.2000,0.1250,0.0750,0.2000,0.0625,0.1375,1.0000,0.5000,0.5000,1.0,1.0,0.0
1,When was the first case of COVID-19 identified?,"[{'text': 'Abstract: In the WHO European Region, COVID-19 surveillance was i...",The answer to the question cannot be directly found in the given context. Ho...,0.3333,0.2692,0.0641,0.3333,0.0769,0.2564,1.0000,0.2857,0.7143,0.0,1.0,-1.0
2,What is the structure of Hantaan virus?,"[{'text': 'The diameter of hantavirus particles is 80-US210 nm, and the stru...",The structure of Hantaan virus is not explicitly stated in the question. How...,0.4545,0.3000,0.1545,0.4545,0.3000,0.1545,1.0000,1.0000,0.0000,0.0,1.0,-1.0
3,Which viruses may not cause prolonged inflammation due to strong induction o...,[{'text': 'This observation has been replicated in mouse models of infection...,"Based on the given context, it can be inferred that L (which is not explicit...",0.0667,0.4118,-0.3451,0.0667,0.1765,-0.1098,1.0000,0.4286,0.5714,0.0,0.0,0.0
4,What is severe MARS noted for?,[{'text': 'a case definition released by the KSA Ministry of Health in June ...,"The context does not mention ""MARS"" but rather ""MERS"" (Middle East Respirato...",0.3333,0.3636,-0.0303,0.1111,0.0455,0.0656,0.3333,0.1250,0.2083,0.0,1.0,-1.0
5,What was the focus of the study?,"[{'text': 'study began.', 'index': 849, 'score': 0.0323, 'dense_score': 0.80...","Based on the given context, it can be inferred that the focus of the study w...",0.5714,0.2941,0.2773,0.2857,0.1765,0.1092,0.5000,0.6000,-0.1000,0.0,1.0,-1.0
6,How many people did SARS-CoV infect?,[{'text': 'SARS-CoV caused 8098 reported cases and 774 deaths in 37 countrie...,"Based on the given context, the answer is not explicitly stated as a single ...",0.8182,0.0667,0.7515,0.4545,0.0667,0.3878,0.5556,1.0000,-0.4444,0.0,1.0,-1.0
7,What contributed to a large part of mammalian genomic sequence?,[{'text': 'Since it first became possible to compare the sequences of comple...,"The question seems incomplete, but based on the provided context, it appears...",1.0000,0.0556,0.9444,0.3846,0.0556,0.3290,0.3846,1.0000,-0.6154,0.0,1.0,-1.0
8,What animal models exist for both the asymptomatic carriage of PUUV and SNV?,[{'text': 'Testing models of pathogenesis can be done more effectively when ...,The animal models that exist for both the asymptomatic carriage of PUUV and ...,0.1538,0.1579,-0.0041,0.1538,0.0526,0.1012,1.0000,0.3333,0.6667,1.0,1.0,0.0
9,What are exhibited in the two phases?,[{'text': 'Examples of phase changes are when water solidifies at freezing p...,"Based on the context, it seems that the question is referring to the phases ...",0.8000,0.1333,0.6667,0.0000,0.0667,-0.0667,0.0000,0.5000,-0.5000,0.0,1.0,-1.0


### Aggregate TRACe scores (mean / ground truth / deviation)

,metric,mean_score,mean_ground_truth,std_score,std_ground_truth,mean_abs_error
0,relevance_score,0.5440,0.3190,0.2936,0.1853,0.3110
1,utilization_score,0.3137,0.1996,0.1973,0.1546,0.2132
2,completeness_score,0.6538,0.6187,0.3141,0.2726,0.3326
3,adherence_score,0.2593,0.9630,0.4382,0.1889,0.7037


None

## 10. Compare Configurations

Generate a comparison report across all configurations to see which performs best.

In [14]:
# Generate comparison report
print("Generating comparison report...")
comparison = runner.compare()

print(f"\n✅ Comparison report generated!")
print(f"  Saved to: {experiment_config.report_dir}/comparison.csv")

Generating comparison report...

✅ Comparison report generated!
  Saved to: reports/comparison.csv


In [15]:
# Display comparison
print("\nConfiguration Comparison:")
display(comparison.to_dataframe())


Configuration Comparison:


,config_name,metric,mean_score,mean_ground_truth,std_score,std_ground_truth,mean_abs_error
0,medcpt_semantic_hybrid_rerank_70b,relevance_score,0.5392,0.2956,0.3208,0.2213,0.3424
1,baai_semantic_hybrid_search_70b,relevance_score,0.5440,0.3190,0.2936,0.1853,0.3110
2,semantic_chunking_with_hybrid_search,relevance_score,0.7275,0.2887,0.3259,0.1684,0.4593
3,high_quality_large,relevance_score,0.6994,0.3405,0.2550,0.0713,0.3590
4,medcpt_embedding_with_semantic_chunking,relevance_score,0.7950,0.2887,0.3514,0.1684,0.5229
5,high_quality_small,relevance_score,0.3898,0.2345,0.3718,0.1242,0.3469


In [ ]:
from reporting.base import Report


report = Report.from_jsonl(
    "temp/covidqa_high_recall_hybrid_v2.jsonl",
    config_name="covidqa_high_recall_hybrid_v2",
)

report.display()

report.save_json("reports/covidqa_high_recall_hybrid_v2.json")

AttributeError: type object 'Report' has no attribute 'from_jsonl'

Progress: 102/246 (41.5%) | QPS: 0.13 | ETA: 19.2m | Elapsed: 13.6m

## 11. Summary

The ExperimentRunner provides a complete workflow for:

1. **Configuration-driven data loading** - Specify data source in YAML
2. **Automatic parsing** - Documents parsed using configured parser
3. **Multi-config evaluation** - Test multiple RAG configurations
4. **Parallel execution** - Speed up evaluation with parallel runs
5. **Comprehensive reporting** - Per-query and aggregate metrics
6. **Cross-config comparison** - Identify best performing config

### Key Benefits:

- **Reproducible** - Everything configured in YAML
- **Flexible** - Easy to change data source or parser
- **Scalable** - Parallel execution for faster evaluation
- **Comprehensive** - Detailed metrics and comparisons